In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# وارد کردن کتابخانه‌های تنسورفلو
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print("=== 🧠 فاز ۴: آموزش شبکه عصبی قیفی (Funnel Deep Learning System) ===")

# ۱. بارگذاری داده‌ها از روی فایل اصلی شما
df_master = pd.read_csv('../data/processed/NHANES_Master_Dataset.csv')

# تبدیل ستون جنسیت به صفر و یک برای لایه ورودی شبکه عصبی
if 'Gender' in df_master.columns:
    df_master['Is_Male'] = df_master['Gender'].map({'Male': 1, 'Female': 0})

# ۲. تعیین دقیق ویژگی‌ها بر اساس ستون‌های تصویر اکسل شما
features = [
    'Age', 'Is_Male', 'Weight_kg', 'Height_cm', 'BMI', 
    'Body_Fat_Percent', 'Lean_Mass_kg', 'Activity_Score', 
    'Daily_Protein_Intake_g', 'Genetic_Score'  # نام ستون دقیقاً اصلاح شد
]

X = df_master[features]
y = df_master['Protein_Requirement_g']

# ۳. تقسیم داده به آموزش و تست
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ۴. استانداردسازی داده‌ها (حیاتی برای همگرایی شبکه‌های عصبی)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ۵. طراحی معماری شبکه عصبی قیفی (Funnel Architecture)
model = Sequential([
    # لایه اول (Expansion): افزایش ابعاد از ۱۰ ویژگی به ۶۴ گره برای کشف روابط غیرخطی پیچیده
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.1), # جلوگیری از حفظ کردن داده‌ها
    
    # لایه دوم (Funneling): فشرده‌سازی اطلاعات به ۳۲ گره
    Dense(32, activation='relu'),
    
    # لایه سوم: فشرده‌سازی بیشتر به ۱۶ گره
    Dense(16, activation='relu'),
    
    # لایه خروجی: ۱ گره برای تخمین نهایی میزان پروتئین (رگرسیون خطی)
    Dense(1, activation='linear')
])

# ۶. کامپایل مدل با بهینه‌ساز Adam
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01), loss='mse', metrics=['mae'])

# متوقف کردن خودکار آموزش در صورت بهینه‌شدن خطا برای جلوگیری از اورفیتینگ
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

# ۷. آموزش مدل
print("🔄 در حال آموزش شبکه عصبی قیفی روی داده‌های شما... لطفاً شکیبا باشید.")
history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=150,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0
)

# ۸. ارزیابی روی داده‌های تست
y_pred = model.predict(X_test_scaled).flatten()

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n📊 نتایج نهایی ارزیابی شبکه عصبی قیفی:")
print(f"  • R² Score                    : {r2:.4f}")
print(f"  • Mean Absolute Error (MAE)   : {mae:.4f} grams")
print(f"  • Root Mean Squared Error (RMSE): {rmse:.4f} grams")

# ۹. رسم و ذخیره نمودار کاهش خطا در پوشه گزارش‌ها
os.makedirs('reports/figures', exist_ok=True)
plt.figure(figsize=(6, 4))
plt.plot(history.history['loss'], label='Train Loss (MSE)', color='#1f77b4')
plt.plot(history.history['val_loss'], label='Validation Loss (MSE)', color='#ff7f0e')
plt.title('Neural Network Funnel Loss Convergence', fontweight='bold', pad=12)
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('reports/figures/05_nn_loss_curve.png', dpi=300)
plt.close()

print("\n[✓] نمودار همگرایی با موفقیت در reports/figures/05_nn_loss_curve.png ذخیره شد.")

=== 🧠 فاز ۴: آموزش شبکه عصبی قیفی (Funnel Deep Learning System) ===


C:\Users\hi\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


🔄 در حال آموزش شبکه عصبی قیفی روی داده‌های شما... لطفاً شکیبا باشید.
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step

📊 نتایج نهایی ارزیابی شبکه عصبی قیفی:
  • R² Score                    : 0.9957
  • Mean Absolute Error (MAE)   : 0.7781 grams
  • Root Mean Squared Error (RMSE): 1.1294 grams

[✓] نمودار همگرایی با موفقیت در reports/figures/05_nn_loss_curve.png ذخیره شد.
